# A persistent book tracker

Run the cell below to create or update `books.db` in the kernel's working directory. Your data stays there between runs.

The sample collection contains 23 books with authors (the original three plus 20 more). Ratings are invented examples for practice. Missing sample books are added on each run, without duplicating existing title-and-author pairs.

Existing databases gain an author column automatically. Known sample titles get their missing authors filled in; existing IDs and ratings are preserved. Other books can keep an unknown author.

Each book has an automatically assigned ID. Titles and ratings are required, and ratings must be between 0 and 5. `commit()` saves changes; `closing()` closes the connection even if an error occurs.

The output shows only books rated at least 4. Change the query parameter `(4.0,)` to `(0.0,)` to display all books.


In [ ]:
import sqlite3
from contextlib import closing
from pathlib import Path

# These are example ratings for practice, not published review scores.
sample_books = [
    ("The Hobbit", "J. R. R. Tolkien", 4.8),
    ("Frankenstein", "Mary Shelley", 4.2),
    ("The Time Machine", "H. G. Wells", 3.9),
    ("Pride and Prejudice", "Jane Austen", 4.7),
    ("1984", "George Orwell", 4.6),
    ("Animal Farm", "George Orwell", 4.3),
    ("To Kill a Mockingbird", "Harper Lee", 4.8),
    ("The Great Gatsby", "F. Scott Fitzgerald", 4.1),
    ("Jane Eyre", "Charlotte Brontë", 4.5),
    ("Wuthering Heights", "Emily Brontë", 3.8),
    ("Moby-Dick", "Herman Melville", 3.7),
    ("Little Women", "Louisa May Alcott", 4.4),
    ("Dracula", "Bram Stoker", 4.2),
    ("The Picture of Dorian Gray", "Oscar Wilde", 4.5),
    ("A Christmas Carol", "Charles Dickens", 4.3),
    ("Great Expectations", "Charles Dickens", 4.0),
    ("Treasure Island", "Robert Louis Stevenson", 4.1),
    ("The Adventures of Tom Sawyer", "Mark Twain", 3.9),
    ("Alice's Adventures in Wonderland", "Lewis Carroll", 4.2),
    ("The Secret Garden", "Frances Hodgson Burnett", 4.4),
    ("Anne of Green Gables", "L. M. Montgomery", 4.6),
    ("The War of the Worlds", "H. G. Wells", 4.0),
    ("The Call of the Wild", "Jack London", 4.1),
]

# The database is saved in the notebook kernel's working directory.
database_path = Path("books.db")

with closing(sqlite3.connect(database_path)) as connection:
    connection.execute("""
        CREATE TABLE IF NOT EXISTS books (
            id INTEGER PRIMARY KEY,
            title TEXT NOT NULL,
            author TEXT,
            rating REAL NOT NULL CHECK (rating BETWEEN 0 AND 5)
        )
    """)

    # Upgrade databases created by the earlier version of this notebook.
    columns = {column[1] for column in connection.execute("PRAGMA table_info(books)")}
    if "author" not in columns:
        connection.execute("ALTER TABLE books ADD COLUMN author TEXT")

    for title, author, rating in sample_books:
        # Fill missing authors without changing existing IDs or ratings.
        connection.execute(
            "UPDATE books SET author = ? WHERE title = ? AND author IS NULL",
            (author, title),
        )

        # Match both title and author so rerunning does not add duplicates.
        connection.execute(
            """
            INSERT INTO books (title, author, rating)
            SELECT ?, ?, ?
            WHERE NOT EXISTS (
                SELECT 1 FROM books WHERE title = ? AND author = ?
            )
            """,
            (title, author, rating, title, author),
        )

    # Save changes before closing the connection.
    connection.commit()

    # Find books with a rating of at least 4, highest-rated first.
    rows = connection.execute(
        """
        SELECT id, title, author, rating FROM books
        WHERE rating >= ?
        ORDER BY rating DESC, id
        """,
        (4.0,),
    ).fetchall()

    for book_id, title, author, rating in rows:
        print(f"{book_id}. {title} by {author or 'Unknown author'}: {rating:.1f}/5")
